### 提取目录结构

#### 初步文档解析
- 采用pageindex中的方法：先去检测存在目录结构的页面，然后利用大模型提取其中的目录信息
- 不能很好的解析扫描文档

In [8]:
from io import BytesIO
import tiktoken
import PyPDF2
import pymupdf
import os

def get_page_tokens(pdf_path, model="gpt-4o-2024-11-20", pdf_parser="PyMuPDF"):
    enc = tiktoken.encoding_for_model(model)
    if pdf_parser == "PyPDF2":
        pdf_reader = PyPDF2.PdfReader(pdf_path)
        page_list = []
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            page_text = page.extract_text()
            token_length = len(enc.encode(page_text))
            page_list.append((page_text, token_length))
        return page_list
    elif pdf_parser == "PyMuPDF":
        if isinstance(pdf_path, BytesIO):
            pdf_stream = pdf_path
            doc = pymupdf.open(stream=pdf_stream, filetype="pdf")
        elif isinstance(pdf_path, str) and os.path.isfile(pdf_path) and pdf_path.lower().endswith(".pdf"):
            doc = pymupdf.open(pdf_path)
        page_list = []
        for page in doc:
            page_text = page.get_text()
            token_length = len(enc.encode(page_text))
            page_list.append((page_text, token_length))
        return page_list
    else:
        raise ValueError(f"Unsupported PDF parser: {pdf_parser}")

#### 按照mineru读取页面数据

In [9]:
import json
import tiktoken

def get_page_content_tokens(pdf_path, model="gpt-4o-2024-11-20"):
    enc = tiktoken.encoding_for_model(model)
    page_list = []
    with open(pdf_path, 'r') as f:
        data = json.load(f)
        page_content = ""
        current_page_index = 0
        for item in data:
            if item.get("page_idx") != current_page_index:
                token_length = len(enc.encode(page_content))
                page_list.append((page_content, token_length))
                page_content = ""
                current_page_index = item.get("page_idx")
            if item['type'] == 'text':
                if item.get("text_level"):
                    page_content += "#"*item['text_level'] + " " + item['text'] + "\n\n "
                else:
                    page_content += item['text'] + "\n\n "
            elif item['type'] == 'list' and item.get("sub_type") == "text":
                page_content += "\n".join(item['list_items']) + "\n\n "
    return page_list

test_path = "/home/hp-2/Agent_Platform/MinerU/demo/output/demo3/hybrid_auto/demo3_content_list.json"
page_list = get_page_content_tokens(test_path)

#### 检测存在目录的页面

In [10]:
import re
import openai
import logging
import time
import os
from json_repair import repair_json

Qwen_API_KEY = os.getenv("Qwen_API_KEY", "sk-proj-1234567890")
Qwen_URL = os.getenv("Qwen_URL", "http://localhost:11434/v1/")
Qwen_MODEL = os.getenv("Qwen_MODEL", "nothink_qwen3_14b")

def ChatGPT_API(prompt, model=Qwen_MODEL, api_key=Qwen_API_KEY, chat_history=None):
    max_retries = 10
    client = openai.OpenAI(api_key=api_key, base_url=Qwen_URL)
    for i in range(max_retries):
        try:
            if chat_history:
                messages = chat_history
                messages.append({"role": "user", "content": prompt})
            else:
                messages = [{"role": "user", "content": prompt}]
            
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                extra_body={"enable_thinking": False},
            )
   
            return response.choices[0].message.content
        except Exception as e:
            print('************* Retrying *************')
            logging.error(f"Error: {e}")
            if i < max_retries - 1:
                time.sleep(1)  # Wait for 1秒 before retrying
            else:
                logging.error('Max retries reached for prompt: ' + prompt)
                return "Error"


def toc_detector_single_page(content, model=Qwen_MODEL):
    prompt = f"""
        ### Role
        你是一个专业的文档结构分析专家。你的任务是判断给定的文本片段是否为文档的“目录页 (Table of Contents)”。

        ### Input Data
        文本内容:
        {content}

        ### Analysis Criteria (判定标准)
        请基于以下特征进行分析：

        1. **核心特征 (必须具备)**:
        - **映射关系**: 每一行通常由“章节标题”和“页码/索引”组成，页码非必须。
        - **列表结构**: 文本呈现垂直排列的列表形态，而非连续的段落语句。
        - **层级标识**: 包含章节序号（如 I, II, A, B, 1, 2...）或缩进结构。

        2. **排除项 (必须排除)**:
        - **叙述性文本**: 即使包含“第一章”、“共分三部分”等字眼，如果它们存在于完整的句子或自然段落中（例如：“本书第一章介绍了X，第二章分析了Y”），**判定为 NO**。
        - **非主目录列表**: “图表目录 (List of Figures)”、“表格目录 (List of Tables)”、“缩略语表”等，**判定为 NO**（除非它们与主目录混合在一起）。

        ### Output Format
        请仅返回以下 JSON 格式，不要包含 markdown 标记或额外解释：

        {{
            "thinking": "简要分析文本的结构特征。1. 是否检测到连续的标题？ 2. 是否排除了叙述性语句？ 3. 是否排除了图表目录？",
            "toc_detected": "yes" | "no"
        }}
    """

    response = ChatGPT_API(model=model, prompt=prompt)
    # print('response', response)
    json_content = repair_json(response, return_objects=True)    
    return json_content['toc_detected']


def find_toc_pages(start_page_index, page_list, toc_check_page_num=15, logger=None):
    print('start find_toc_pages')
    last_page_is_yes = False
    toc_page_list = []
    i = start_page_index
    
    while i < len(page_list):
        # Only check beyond max_pages if we're still finding TOC pages
        if i >= toc_check_page_num and not last_page_is_yes:
            break
        detected_result = toc_detector_single_page(page_list[i][0])
        if detected_result == 'yes':
            if logger:
                logger.info(f'Page {i} has toc')
            toc_page_list.append(i)
            last_page_is_yes = True
        elif detected_result == 'no' and last_page_is_yes:
            if logger:
                logger.info(f'Found the last page with toc: {i-1}')
            break
        i += 1
    
    if not toc_page_list and logger:
        logger.info('No toc found')
        
    return toc_page_list


def toc_extractor(page_list, toc_page_list, model):
    def transform_dots_to_colon(text):
        text = re.sub(r'\.{5,}', ': ', text)
        # Handle dots separated by spaces
        text = re.sub(r'(?:\. ){5,}\.?', ': ', text)
        return text
    
    toc_content = ""
    for page_index in toc_page_list:
        toc_content += page_list[page_index][0]
    toc_content = transform_dots_to_colon(toc_content)
    
    return {
        "toc_content": toc_content
    }


def check_toc(page_list, toc_check_page_num=15, model=Qwen_MODEL):
    toc_page_list = find_toc_pages(start_page_index=0, page_list=page_list, toc_check_page_num=15)
    if len(toc_page_list) == 0:
        print('no toc found')
        return {'toc_content': None, 'toc_page_list': []}
    else:
        print('toc found')
        toc_json = toc_extractor(page_list, toc_page_list, model)
        return {'toc_content': toc_json['toc_content'], 'toc_page_list': toc_page_list}

check_toc_result = check_toc(page_list)
print(check_toc_result)

start find_toc_pages
no toc found
{'toc_content': None, 'toc_page_list': []}


#### 提取目录

In [11]:
import copy
import math
import json

CHATGPT_API_KEY = os.getenv("CHATGPT_API_KEY", "sk-proj-1234567890")
BASE_URL = os.getenv("BASE_URL", "http://localhost:11434/v1/")
MODEL = os.getenv("MODEL", "nothink_qwen3_14b")

def count_tokens(text, model=None):
    if not text:
        return 0
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        # If model is not recognized, use cl100k_base encoding (same as GPT-4/GPT-3.5-turbo)
        enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    return len(tokens)

def page_list_to_group_text(page_contents, token_lengths, max_tokens=20000, overlap_page=1):    
    num_tokens = sum(token_lengths)
    
    if num_tokens <= max_tokens:
        # merge all pages into one text
        page_text = "".join(page_contents)
        return [page_text]
    
    subsets = []
    current_subset = []
    current_token_count = 0

    expected_parts_num = math.ceil(num_tokens / max_tokens)
    average_tokens_per_part = math.ceil(((num_tokens / expected_parts_num) + max_tokens) / 2)
    
    for i, (page_content, page_tokens) in enumerate(zip(page_contents, token_lengths)):
        if current_token_count + page_tokens > average_tokens_per_part:

            subsets.append(''.join(current_subset))
            # Start new subset from overlap if specified
            overlap_start = max(i - overlap_page, 0)
            current_subset = page_contents[overlap_start:i]
            current_token_count = sum(token_lengths[overlap_start:i])
        
        # Add current page to the subset
        current_subset.append(page_content)
        current_token_count += page_tokens

    # Add the last subset if it contains any pages
    if current_subset:
        subsets.append(''.join(current_subset))
    
    print('divide page_list to groups', len(subsets))
    return subsets

def ChatGPT_API_with_finish_reason(model, prompt, api_key=CHATGPT_API_KEY, chat_history=None):
    max_retries = 10
    client = openai.OpenAI(api_key=api_key, base_url=BASE_URL)
    for i in range(max_retries):
        try:
            if chat_history:
                messages = chat_history
                messages.append({"role": "user", "content": prompt})
            else:
                messages = [{"role": "user", "content": prompt}]
            
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                extra_body={"enable_thinking": False},
            )
            if response.choices[0].finish_reason == "length":
                return response.choices[0].message.content, "max_output_reached"
            else:
                return response.choices[0].message.content, "finished"

        except Exception as e:
            print('************* Retrying *************')
            logging.error(f"Error: {e}")
            if i < max_retries - 1:
                time.sleep(1)  # Wait for 1秒 before retrying
            else:
                logging.error('Max retries reached for prompt: ' + prompt)
                return "Error"

def generate_toc_init(part, model=None):
    prompt = """
    # Role
    你是一位文档结构与版面分析专家。你的任务是从提供的文本中精准提取目录（Table of Contents）层级结构，并将其转换为标准的 JSON 格式。

    # Constraints
    1. **严格输出 JSON**：直接返回最终的 JSON 列表，不要包含 Markdown 标记（如 ```json），不要输出任何解释性文字。
    2. **排除干扰项**：忽略文本中的页眉、页脚、水印或与目录内容无关的当前页面页码。
    3. **层级一致性**：根据标题的格式（如字体大小、缩进、序号格式）逻辑判断章节的父子关系。

    # Field Definitions
    请按照以下定义提取字段：

    - **structure** (string): 
        - 章节的层级索引系统（如 "1", "1.1", "1.1.1"）。
        - 即使原文没有显式序号，也必须根据逻辑层级自动生成该索引。

    - **title** (string): 
        - 提取章节标题文本。
        - **处理规则**：去除首尾空格，将连续的内部空格合并为一个空格。保留原文的文字内容，不要改写。

    - **page** (string | None): 
        - **定义**：提取该目录条目指向的目标页码（通常位于行尾）。
        - **关键区别**：**严禁**提取当前页面的物理页码（通常单独出现在底部或顶部）。
        - **识别特征**：目录页码通常与标题在同一行，中间可能有省略号（......）或长空格分隔。
        - **缺失处理**：如果该条目没有对应的引导页码，必须返回 `None`。

    # Output Format
    [
        {"structure": <结构索引, "x.x.x"> (字符串), "title": <章节标题，保持原始标题>, "page": <page number or None>},
    ]
    """

    prompt = prompt + '\nGiven text\n:' + part
    # print(prompt)
    response, finish_reason = ChatGPT_API_with_finish_reason(model=model, prompt=prompt)
    # print(response)

    if finish_reason == 'finished':
         return repair_json(response, return_objects=True) 
    else:
        raise Exception(f'finish reason: {finish_reason}')

def generate_toc_continue(toc_content, part, model=MODEL):
    print('start generate_toc_continue')
    prompt = """
    你是提取层次结构树结构的专家。
    你将获得前一部分的树形结构和当前部分的文本。
    你的任务是继续从前一部分的树形结构，将当前部分包含进来。

    # Field Definitions
    请按照以下定义提取字段：
    - **structure** (string): 
        - 章节的层级索引系统（如 "1", "1.1", "1.1.1"）。
        - 即使原文没有显式序号，也必须根据逻辑层级自动生成该索引。

    - **title** (string): 
        - 提取章节标题文本。
        - **处理规则**：去除首尾空格，将连续的内部空格合并为一个空格。保留原文的文字内容，不要改写。

    - **page** (string | None): 
        - **定义**：提取该目录条目指向的目标页码（通常位于行尾）。
        - **关键区别**：**严禁**提取当前页面的物理页码（通常单独出现在底部或顶部）。
        - **识别特征**：目录页码通常与标题在同一行，中间可能有省略号（......）或长空格分隔。
    """

    prompt = prompt + '\nGiven text\n:' + part + '\nPrevious tree structure\n:' + json.dumps(toc_content, indent=2)
    response, finish_reason = ChatGPT_API_with_finish_reason(model=model, prompt=prompt)
    # print(response)
    if finish_reason == 'finished':
        return repair_json(response, return_objects=True) 
    else:
        raise Exception(f'finish reason: {finish_reason}')

def process_no_toc(page_list, toc_page_list, model=MODEL, logger=None):
    page_contents=[]
    token_lengths=[]
    for page_index in toc_page_list:
        page_text = f"<chunk>\n{page_list[page_index][0]}\n</chunk>\n\n"
        page_contents.append(page_text)
        token_lengths.append(count_tokens(page_text, model))
    group_texts = page_list_to_group_text(page_contents, token_lengths)

    toc_with_page_number= generate_toc_init(group_texts[0], model)
    for group_text in group_texts[1:]:
        toc_with_page_number_additional = generate_toc_continue(toc_with_page_number, group_text, model)    
        toc_with_page_number.extend(toc_with_page_number_additional)
    
    return toc_with_page_number


# 有目录但是没有目录页码
toc_page_list = check_toc_result["toc_page_list"]
if len(toc_page_list) > 0:
    toc_with_page_number = process_no_toc(page_list, toc_page_list, model=MODEL)
else:
    print("没有目录页码")
    toc_with_page_number = process_no_toc(page_list, [i for i in range(len(page_list))][:5], model=MODEL)


没有目录页码


In [12]:
toc_with_page_number

[{'structure': '1', 'title': 'Introduction', 'page': None},
 {'structure': '2',
  'title': 'Preliminaries: TAPAS for Table Encoding',
  'page': None},
 {'structure': '3',
  'title': 'TABLEFORMER: Robust Structural Table Encoding',
  'page': None},
 {'structure': '4', 'title': 'Experimental Setup', 'page': None},
 {'structure': '4.1', 'title': 'Datasets and Evaluation', 'page': None},
 {'structure': '4.2', 'title': 'Baselines', 'page': None},
 {'structure': '4.3',
  'title': 'Perturbing Tables as Augmented Data',
  'page': None},
 {'structure': '5', 'title': 'Experiments and Results', 'page': None}]

#### 目录交叉验证
- 直接使用mineru的中间结果提取存在level级别的目录

In [13]:
import json
import openai

CHATGPT_API_KEY = os.getenv("CHATGPT_API_KEY", "sk-proj-1234567890")
BASE_URL = os.getenv("BASE_URL", "http://localhost:11434/v1/")
MODEL = os.getenv("MODEL", "nothink_qwen3_14b")

data_path = test_path

TOC = ""
page_number_pattern = re.compile(r'\d+\s*$')
with open(data_path, 'r') as f:
    data = json.load(f)
    for item in data:
        if item['type'] == 'text' and item.get("text_level"):
            TOC = TOC + "\n" + item["text"]
        elif item['type'] == 'text' and page_number_pattern.search(item.get("text")):
            TOC = TOC + "\n" + item["text"]
        elif item['type'] == 'list' and item.get("sub_type") == "text":
            TOC = TOC + "\n" + "\n".join(item['list_items'])

client = openai.OpenAI(api_key=CHATGPT_API_KEY, base_url=BASE_URL)

PROMPT = """
    ### 角色设定
    你是一个高精度的文档目录解析引擎。你的目标是生成一份完美的、结构化的目录树（JSON）。

    ### 输入数据
    Source TOC:
    {toc_with_page_number}

    Context Reference:
    {toc}

    ### 任务输入
    - **Source TOC** : 原始识别的目录，可能存在层级缺失。
    - **Context Reference** : 文档的提取文本，包含真实的标题，但含有大量噪声。

    ### 处理规则 (Step-by-Step)
    1. **交叉验证**：以 `Source TOC` 为基础，利用 `Context Reference` 验证标题是否有缺失。
    2. **逻辑判断**：在决定是否将 `Context Reference` 中的某一行加入目录时，判断其是否具备标题特征（如序号 "1.1"、"Chapter 1" 、"附录"或概括性短语）。**严禁将长段落正文作为标题。**
    3. **层级构建**：确保输出的 JSON 具有正确的嵌套关系（parent-child）。
    4. **数据清洗**：移除所有乱码、重复项及非标题内容。


    ### 输出约束
    - **格式**：仅输出标准的 JSON 数组。
    - **structure**：是一个数字系统，表示目录中章节的层级索引。例如，第一个章节的结构索引为 1，第一个子章节的结构索引为 1.1，第二个子章节的结构索引为 1.2，依此类推。
    - **title**：需要从文本中提取原始标题，只需修正空格不一致的问题。
    返回结果应采用以下格式：
        [
            {"structure": <结构索引, "x.x.x"> (字符串), "title": <章节标题，保持原始标题>},
            ...
        ]    
    直接返回最终 JSON 结构的附加部分。不要输出其他任何内容。
"""

# 检测是否有page_number
page_number = False
for item in toc_with_page_number:
    if item.get("page"):
        page_number = True
        print("存在page_number")
        result = toc_with_page_number
        break

if not page_number:
    print("正在整理目录...")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT.replace("{toc_with_page_number}", str(toc_with_page_number[0])).replace("{toc}", TOC)}],
        temperature=0.6,
        extra_body={"enable_thinking": False},
    )
    result = response.choices[0].message.content
    result = repair_json(result, return_objects=True)

正在整理目录...


In [14]:
def print_toc_items(items, indent=0):
    """
    递归打印目录树结构
    
    Args:
        items: 目录项列表
        indent: 缩进级别，用于视觉层次
    """
    for item in items:
        # 打印当前项
        item_id = item.get('structure', '')
        item_title = item.get('title', '')
        print("  " * indent + f"{item_id} {item_title}")
        
        # 如果有子项，递归处理
        if item.get("children"):
            print_toc_items(item["children"], indent + 1)

# 使用示例
print_toc_items(result)

1 Introduction
2 Preliminaries: TAPAS for Table Encoding
3 TABLEFORMER: Robust Structural Table Encoding
4 Experimental Setup
4.1 Datasets and Evaluation
4.2 Baselines
4.3 Perturbing Tables as Augmented Data
5 Experiments and Results
5.1 Main Results
5.2 Perturbation Results
5.3 Model Size Comparison
5.4 Analysis of TABLEFORMER Submodules
5.5 Comparison of TABLEFORMER and Perturbed Data Augmentation
5.6 Attention Bias Ablation Study
5.7 Limitations of TABLEFORMER
6 Other Related Work
7 Conclusion
